# Comprehensive Attack Strategy Report for Gaitkeeper Project

## Executive Summary

This report analyzes all strategic choices for attacking gait recognition systems, comparing attack methods, target models, and tactical approaches. Based on technical feasibility, real-world robustness, and project constraints, I recommend: **Context-aware Adversarial Patch with multi-objective optimization targeting YOLOv8-seg person segmentation using Expectation over Transformation (EoT).**

---

# Part 1: Target Model Selection

## Available Model Types for Gait Recognition

### Option A: Multi-Stage Gait Recognition Models

**Examples:** GaitSet, GaitPart, GaitGL

**Architecture:**

1. Person detection/segmentation → 2. Silhouette extraction → 3. Feature extraction → 4. Gait classification

**Pros:**

- Most realistic (actual gait recognition systems)
- Can measure attack success at multiple stages
- If you break stage 1, everything fails (cascading failure)

**Cons:**

- Complex to set up and run
- May require specific datasets (CASIA-B, OU-MVLP)
- Harder to debug when things go wrong
- Limited documentation for some models

**Attack Surface:**

- Stage 1 (person segmentation) ← **YOUR TARGET**
- Stage 2 (silhouette quality)
- Stage 3 (feature extraction)

---

### Option B: Person Segmentation Models Only

**Examples:** YOLOv8-seg, Mask R-CNN, DeepLabV3+, Segment Anything Model (SAM)

**Architecture:**
Input image → CNN backbone → Segmentation head → Person mask output

**Pros:**

- Well-documented and easy to use
- Available on Hugging Face with pre-trained weights
- Fast inference (can process videos efficiently)
- Breaking this breaks ALL downstream gait analysis
- Easier to evaluate (clear IoU metrics)

**Cons:**

- Not specifically designed for gait (though used in gait pipelines)
- Need to separately verify it affects gait recognition

**Attack Surface:**

- CNN feature extraction layers
- Segmentation boundaries
- Confidence scores

---

### Option C: Object Detection Models

**Examples:** YOLO, Faster R-CNN, RetinaNet

**Architecture:**
Input image → CNN backbone → Detection head → Bounding boxes + class probabilities

**Pros:**

- Extremely well-documented
- Fast and efficient
- If you can't detect the person, gait analysis is impossible

**Cons:**

- Only gives bounding boxes, not pixel-level segmentation
- Gait systems typically need silhouettes, not just boxes
- Less informative for your specific use case

---

## **RECOMMENDATION: YOLOv8-seg (Person Segmentation)**

### Setup Code:

```python
from ultralytics import YOLO

# Load pre-trained model
model = YOLO('yolov8n-seg.pt')  # nano version (fastest)
# or yolov8s-seg.pt (small), yolov8m-seg.pt (medium)

# Run on video
results = model('walking_video.mp4')

# Extract person segmentations
for result in results:
    masks = result.masks  # Segmentation masks
    boxes = result.boxes  # Bounding boxes
    conf = result.boxes.conf  # Confidence scores
```

### Backup Option: Mask R-CNN

If YOLOv8-seg proves too difficult to attack, Mask R-CNN has more attack literature and is slightly older (more studied).

---

# Part 2: Attack Method Comparison

## Attack Method 1: Fast Gradient Sign Method (FGSM)

### How It Works:

```
perturbation = ε × sign(∇_x Loss(y_true, model(x)))
adversarial_image = original_image + perturbation
```

### Pros:

- ✅ **Extremely simple** (10 lines of code)
- ✅ **Fast** (one forward pass, one backward pass)
- ✅ **Good baseline** (prove concept works)
- ✅ **Well understood** (lots of literature)

### Cons:

- ❌ **Not robust to transformations** (rotation, scaling breaks it)
- ❌ **Works on whole image** (can't localize to clothing)
- ❌ **Digital-only success** (fails in physical world ~80% of time)
- ❌ **No context awareness** (doesn't consider viewing angle, lighting)

### Use Case:

**Proof of concept only.** Start here to verify your pipeline works, then move to more robust methods.

**Verdict:** Too brittle for physical-world clothing application.

---

## Attack Method 2: Basic Adversarial Patch

### How It Works:

```
# Optimize a localized patch instead of whole image
patch = initialize_random_patch(size=(300, 300))

for iteration in training:
    # Apply patch to random locations on image
    adversarial_image = apply_patch(original_image, patch, location)
    loss = compute_loss(model(adversarial_image))
    patch = patch + lr × sign(∇_patch loss)
```

### Pros:

- ✅ **Localized** (can place on torso specifically)
- ✅ **Printable** (fixed-size patch can be manufactured)
- ✅ **Tested in literature** (many successful physical implementations)
- ✅ **Better than FGSM** (some spatial robustness)

### Cons:

- ❌ **Still fragile** (lighting/angle changes reduce effectiveness)
- ❌ **No explicit robustness guarantees**
- ❌ **Binary placement** (either patch is there or not, no blending)

**Verdict:** Better than FGSM, but needs enhancement with EoT.

---

## Attack Method 3: Adversarial Patch + EoT (Expectation over Transformation)

### How It Works:

```python
patch = initialize_random_patch(size=(300, 300))

for iteration in training:
    total_loss = 0

    # Sample multiple transformations
    for transformation in sample_transformations(num_samples=20):
        # Apply random rotation, scale, brightness, etc.
        transformed_patch = transform(patch, transformation)

        adversarial_image = apply_patch(original_image, transformed_patch)
        loss = compute_loss(model(adversarial_image))
        total_loss += loss

    # Update patch based on average loss over all transformations
    patch = patch + lr × sign(∇_patch (total_loss / num_samples))
```

### Transformations Included:

- Rotation: ±20°
- Scale: 0.85-1.15×
- Translation: ±50 pixels
- Brightness: 0.7-1.3×
- Contrast: 0.8-1.2×
- Perspective warp (3D effects)
- Gaussian blur (motion/camera)
- Gaussian noise (sensor noise)

### Pros:

- ✅ **Robust to real-world conditions** (optimized for variation)
- ✅ **Higher physical success rate** (~60-70% vs ~20% for basic patch)
- ✅ **Handles fabric wrinkles** (via perspective warp)
- ✅ **Works across viewing angles** (via rotation)
- ✅ **Multiple camera distances** (via scale)

### Cons:

- ❌ **Computationally expensive** (20× slower than basic patch)
- ❌ **Requires more training time**
- ❌ **More complex to implement**

**Verdict:** Gold standard for physical adversarial examples. Required for your use case.

---

## Attack Method 4: FashionAdv (Fashion-Guided Adversarial Attack)

### How It Works:

Specifically designed for attacking person segmentation via clothing patterns.

```python
# FashionAdv pipeline
1. Segment person and clothing regions
2. Generate adversarial pattern constrained to clothing texture space
3. Optimize pattern to:
   - Confuse segmentation boundaries
   - Stay within realistic clothing textures
   - Maintain "fashionable" appearance
4. Apply smoothness constraints for printability
```

### Key Innovation:

Uses **fashion dataset priors** (patterns from real clothing) to constrain optimization, making patterns look more natural.

### Pros:

- ✅ **Designed specifically for clothing** (perfect fit for your project)
- ✅ **More aesthetic** (patterns look like actual fashion)
- ✅ **Proven effectiveness** (published at CVPR)
- ✅ **Segmentation-focused** (exactly your target)

### Cons:

- ❌ **More complex implementation** (need fashion dataset, texture priors)
- ❌ **Requires fashion dataset** (extra data collection)
- ❌ **Less literature/support** (newer method, fewer tutorials)
- ❌ **May be overkill** (if aesthetics aren't priority)

**Verdict:** Excellent if you have time and want aesthetic patterns. Otherwise, EoT patches are simpler and nearly as effective.

---

## Attack Method 5: Iterative Methods (PGD, C&W)

### Projected Gradient Descent (PGD):

```python
# Like FGSM but iterative
adversarial = original_image
for step in range(num_iterations):
    gradient = compute_gradient(adversarial)
    adversarial = adversarial + α × sign(gradient)
    # Project back to valid range
    adversarial = clip(adversarial, original_image - ε, original_image + ε)
```

### Carlini & Wagner (C&W):

More sophisticated optimization using L2 distance minimization.

### Pros:

- ✅ **Stronger attacks** (harder for model to defend against)
- ✅ **Can be more subtle** (smaller perturbations)
- ✅ **Better optimization** (finds better local minima)

### Cons:

- ❌ **Much slower** (10-100× slower than FGSM)
- ❌ **Works on whole image** (not patch-localized without modification)
- ❌ **Still digital-focused** (need EoT for physical robustness)

**Verdict:** Overkill. Better to use simpler methods with EoT.

---

## **RECOMMENDED ATTACK METHOD: Adversarial Patch + EoT**

---

# Part 3: Attack Objective Comparison

## Objective A: Maximize Uncertainty (Entropy Maximization)

### Goal:

Make the model unsure whether there's a person or not.

### Loss Function:

```python
def uncertainty_loss(model_output):
    # Maximize entropy of predictions
    probs = softmax(model_output)
    entropy = -sum(probs × log(probs))
    loss = -entropy  # Negative because we want to maximize
    return loss
```

### What This Achieves:

- Model outputs: "50% person, 50% background" (confused)
- Confidence scores drop dramatically
- Segmentation boundaries become fuzzy

### Pros:

- ✅ **More robust** (works even if model is partially correct)
- ✅ **Easier to achieve** (don't need to force specific wrong answer)
- ✅ **Cascading failure** (uncertainty propagates to gait classifier)
- ✅ **Natural goal** (aligns with "confusing" the system)

### Cons:

- ❌ **Person might still be detected** (just with low confidence)
- ❌ **Some gait systems might have uncertainty threshold** (still processes if >30% confident)

### Success Criteria:

- Average confidence drops from 0.95 to <0.6
- Entropy increases by >50%

---

## Objective B: Maximize Misclassification (Targeted Attack)

### Goal:

Make the model confidently predict WRONG class (e.g., "this is definitely NOT a person").

### Loss Function:

```python
def misclassification_loss(model_output, target_class="background"):
    # Minimize loss for target class (or maximize loss for correct class)
    loss = CrossEntropyLoss(target=target_class, prediction=model_output)
    return loss  # Want this HIGH

    # OR untargeted (just be wrong):
    loss = -CrossEntropyLoss(target=correct_class, prediction=model_output)
    return loss
```

### What This Achieves:

- Model outputs: "95% confident this is background/furniture" (wrong but confident)
- Person is not detected at all
- Segmentation mask is completely wrong

### Pros:

- ✅ **Complete evasion** (person not detected at all)
- ✅ **Foolproof** (if model thinks you're furniture, gait analysis impossible)
- ✅ **Strong attack** (harder to defend against)

### Cons:

- ❌ **Harder to achieve** (requires pushing all the way to wrong answer)
- ❌ **Less robust** (small changes might bring it back to "person")
- ❌ **May look more obvious** (need stronger perturbations)

### Success Criteria:

- Person detection rate drops to <20%
- Model confidently predicts wrong class (>0.8 confidence in "not person")

---

## Objective C: IoU Reduction (Segmentation Disruption)

### Goal:

Make the segmentation mask NOT overlap with actual person silhouette.

### Loss Function:

```python
def iou_reduction_loss(predicted_mask, ground_truth_mask):
    intersection = (predicted_mask * ground_truth_mask).sum()
    union = predicted_mask.sum() + ground_truth_mask.sum() - intersection
    iou = intersection / (union + 1e-6)
    loss = iou  # We want this LOW, so minimize it
    return loss
```

### What This Achieves:

- Predicted person mask doesn't align with actual person
- Arms might be classified as background, torso as furniture
- Gait features become nonsensical

### Pros:

- ✅ **Directly targets silhouette** (exactly what gait needs)
- ✅ **Measurable** (clear metric: IoU)
- ✅ **Cascading effect** (wrong silhouette → wrong gait features)

### Cons:

- ❌ **Might still detect person** (just segment it wrong)
- ❌ **Some gait systems might be robust to segmentation errors**

### Success Criteria:

- IoU drops from 0.9 to <0.4
- Segmentation boundaries are significantly wrong

---

## Objective D: Confidence Suppression

### Goal:

Lower the model's confidence in its prediction (regardless of what it predicts).

### Loss Function:

```python
def confidence_suppression_loss(model_output):
    max_confidence = torch.max(softmax(model_output))
    loss = -max_confidence  # Want confidence LOW
    return loss
```

### What This Achieves:

- Model is uncertain about everything
- Low confidence triggers might skip person in downstream processing

### Pros:

- ✅ **Simple** (just suppress maximum prediction)
- ✅ **Works well combined with other objectives**

### Cons:

- ❌ **Weak alone** (model might still predict correctly, just with less confidence)

---

## **RECOMMENDED OBJECTIVE: Multi-Objective Combination**

### Why Combine Multiple Objectives?

**Attack multiple vulnerabilities simultaneously:**

```python
def combined_loss(model_output, ground_truth_mask, predicted_mask):
    # Primary: Maximize uncertainty
    L_uncertainty = -calculate_entropy(model_output)

    # Secondary: Reduce IoU
    L_iou = calculate_iou(predicted_mask, ground_truth_mask)

    # Tertiary: Suppress confidence
    L_confidence = torch.max(softmax(model_output))

    # Bonus: Edge disruption (disrupt contours specifically)
    L_edge = calculate_edge_loss(predicted_mask, ground_truth_mask)

    # Weighted combination
    total_loss = (
        0.4 × L_uncertainty +    # Most important
        0.3 × L_iou +            # Very important for gait
        0.2 × L_confidence +     # Helpful bonus
        0.1 × L_edge             # Target contour detection specifically
    )

    return total_loss
```

### Why This Works Best:

- ✅ **Redundancy** (if one objective fails, others still work)
- ✅ **Comprehensive attack** (hits multiple pipeline stages)
- ✅ **Robustness** (harder to defend against)
- ✅ **Flexibility** (can adjust weights based on results)

---

# Part 4: Context-Aware vs Non-Context-Aware

## Non-Context-Aware Generation

### Approach:

```python
# Generate pattern in isolation
pattern = optimize_pattern(model, loss_function)

# Apply to clothing later
adversarial_image = overlay_pattern(original_image, pattern)
```

### Pros:

- ✅ **Simpler** (don't need to model transformations)
- ✅ **Faster** (no transformation sampling)

### Cons:

- ❌ **Fails in physical world** (~80% failure rate)
- ❌ **Doesn't account for fabric wrinkles, lighting, angles**
- ❌ **Optimized for perfect conditions only**

---

## Context-Aware Generation

### Approach:

```python
# Generate pattern while considering real-world conditions
for iteration in training:
    for video_frame in dataset:
        # Get torso mask for this specific frame
        torso_mask = segment_torso(video_frame)

        # Apply transformations
        transformed_pattern = apply_random_transforms(pattern)

        # Overlay on actual torso location
        adversarial_frame = overlay_on_mask(video_frame, transformed_pattern, torso_mask)

        # Optimize
        loss = compute_loss(model(adversarial_frame))
        update_pattern(loss)
```

### What Makes It "Context-Aware":

- Uses actual body shapes and poses from dataset
- Applies pattern to REAL torso locations (not arbitrary placement)
- Accounts for body curvature (3D → 2D projection)
- Optimizes for specific context (walking videos, not static images)

### Pros:

- ✅ **Much more robust** (~60-70% physical success rate)
- ✅ **Realistic optimization** (trained on actual use case)
- ✅ **Better silhouette disruption** (knows where torso actually is)

### Cons:

- ❌ **Requires dataset of walking videos**
- ❌ **Requires torso segmentation pipeline**
- ❌ **More complex implementation**

---

## **RECOMMENDATION: Context-Aware with EoT**

This is non-negotiable for physical-world success. Your project explicitly states physical clothing is the goal, so context-awareness is mandatory.

---

# Part 5: Complete Recommended Strategy

## Final Attack Configuration

### Target Model:

**YOLOv8-seg (person segmentation)**

**Rationale:**

- Easy to setup and use
- Well-documented
- Actually used in real surveillance
- Fast enough for video processing
- Clear evaluation metrics

---

### Attack Method:

**Adversarial Patch with Expectation over Transformation (EoT)**

**Rationale:**

- Proven physical-world robustness
- Patch-based (printable on clothing)
- EoT handles real-world variations
- Well-studied in literature
- Achievable in your timeline

---

### Attack Objective:

**Multi-objective optimization:**

- 40% Uncertainty maximization (primary)
- 30% IoU reduction (secondary)
- 20% Confidence suppression (tertiary)
- 10% Edge disruption (bonus)

**Rationale:**

- Attacks multiple vulnerabilities
- Redundancy if one defense exists
- Specifically targets contour detection (your stated goal)
- Comprehensive pipeline disruption

---

### Generation Strategy:

**Context-aware with EoT**

**Transformations:**

- Rotation: ±20°
- Scale: 0.85-1.15×
- Brightness: 0.7-1.3×
- Contrast: 0.8-1.2×
- Perspective warp: yes
- Motion blur: 0-3 pixels
- Gaussian noise: σ=0.05

**Rationale:**

- Physical robustness is critical
- Your project states physical testing is required
- Context-awareness significantly improves success rate

---

## Implementation Pseudocode

```python
# ============================================
# COMPLETE GAITKEEPER ATTACK PIPELINE
# ============================================

import torch
from ultralytics import YOLO
import cv2
import numpy as np

# === STEP 1: SETUP ===
model = YOLO('yolov8n-seg.pt')
model.eval()

# Initialize random pattern
pattern = torch.rand(300, 300, 3, requires_grad=True)
optimizer = torch.optim.Adam([pattern], lr=0.01)

# === STEP 2: LOAD DATA ===
video_dataset = load_walking_videos('dataset/')  # Your walking videos

# === STEP 3: TRAINING LOOP ===
for epoch in range(100):
    epoch_loss = 0

    for video_frame in video_dataset:
        batch_loss = 0

        # EoT: Sample multiple transformations
        for _ in range(20):  # 20 transformations per frame

            # Apply random transformations
            transformed_pattern = apply_transforms(
                pattern,
                rotation=random.uniform(-20, 20),
                scale=random.uniform(0.85, 1.15),
                brightness=random.uniform(0.7, 1.3),
                contrast=random.uniform(0.8, 1.2),
                perspective_warp=True,
                blur=random.randint(0, 3)
            )

            # Context-aware: Get torso mask
            torso_mask = segment_torso_region(video_frame)

            # Overlay pattern on torso
            adversarial_frame = overlay_pattern_on_mask(
                original=video_frame,
                pattern=transformed_pattern,
                mask=torso_mask
            )

            # Get model prediction
            output = model(adversarial_frame)
            predicted_mask = output.masks
            confidence = output.boxes.conf

            # Multi-objective loss
            L_uncertainty = -calculate_entropy(output)
            L_iou = calculate_iou(predicted_mask, torso_mask)
            L_confidence = torch.max(confidence)
            L_edge = calculate_edge_difference(predicted_mask, torso_mask)

            loss = (
                0.4 * L_uncertainty +
                0.3 * L_iou +
                0.2 * L_confidence +
                0.1 * L_edge
            )

            batch_loss += loss

        # Gradient ascent (maximize loss)
        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Constrain pattern to valid range
        with torch.no_grad():
            pattern.clamp_(0, 1)

        epoch_loss += batch_loss.item()

    # Log progress
    print(f"Epoch {epoch}: Loss = {epoch_loss:.4f}")

    # Save pattern periodically
    if epoch % 10 == 0:
        save_pattern(pattern, f'pattern_epoch_{epoch}.png')

# === STEP 4: EVALUATION ===
# Test on held-out videos
test_accuracy = evaluate_pattern(pattern, test_dataset, model)
print(f"Attack success rate: {test_accuracy:.2%}")

# === STEP 5: PHYSICAL MANUFACTURING ===
# Save best pattern for printing
save_pattern_for_printing(pattern, 'final_pattern.png', dpi=300)
```

---

## Evaluation Metrics

### Digital Testing (Synthetic):

```python
metrics = {
    'person_detection_rate': [],      # Should drop significantly
    'average_confidence': [],          # Should drop below 0.6
    'average_iou': [],                 # Should drop below 0.4
    'false_negative_rate': [],         # Should increase
    'segmentation_boundary_error': []  # Should increase
}

# Success criteria:
# - Detection rate drops by >50%
# - Confidence drops by >30%
# - IoU drops by >40%
```

### Physical Testing (Real-world):

Same metrics, filmed in real conditions.

**Success = <30% performance degradation between digital and physical**

---

## Timeline Breakdown

### Step 1-2: Setup & Baseline

- Install YOLOv8-seg
- Collect/download walking video dataset
- Implement torso segmentation
- Verify model works (baseline accuracy)
- **Deliverable:** Working model on clean videos

### Step 3: Basic Attack

- Implement basic adversarial patch (no EoT)
- Test on static images first
- Verify attack works in digital domain
- **Deliverable:** Proof-of-concept pattern

### Step 4: Add EoT

- Implement transformation pipeline
- Add context-aware overlay
- Train with EoT (20 transformations/frame)
- **Deliverable:** Robust digital attack

### Step 5: Optimization

- Implement multi-objective loss
- Generate 3 pattern variants (noise, geometric, optimized)
- Digital evaluation on synthetic test set
- **Deliverable:** Best pattern candidate

### Step 6-7: Physical Testing

- Print pattern on fabric
- Film test subjects wearing patterns
- Evaluate with same metrics
- **Deliverable:** Physical effectiveness data

### Step 8: Analysis

- Compare digital vs physical performance
- Document findings
- Write final report
- **Deliverable:** Complete project report

---

## Risk Mitigation

### Risk 1: Physical Pattern Doesn't Work

**Mitigation:**

- Test multiple epsilon values (0.05, 0.1, 0.15)
- Generate 3 different pattern types
- Use higher DPI printing (300+ DPI)

### Risk 2: EoT Training Too Slow

**Mitigation:**

- Reduce number of transformations (20 → 10)
- Use smaller model (YOLOv8-nano instead of medium)
- Train on fewer videos initially

### Risk 3: Can't Get Walking Video Dataset

**Mitigation:**

- Film yourselves walking (10-20 videos)
- Use existing datasets (CASIA-B, OU-MVLP)
- Download from YouTube (walking POV videos)

### Risk 4: Pattern Looks Too Obvious

**Mitigation:**

- Add aesthetic constraints (color palette, symmetry)
- Consider FashionAdv approach
- Make pattern larger/more diffuse